# Louis Code 

In [11]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"

taiwan = pd.read_excel(RAW / "taiwan-credit-default" / "default of credit card clients.xls",
                       header=1, index_col="ID")

ImportError: numpy._core.multiarray failed to import

In [ ]:
# Identify payment-status columns
pay_cols = [
    col for col in taiwan.columns
    if str(col).upper() in {f"PAY_{i}" for i in range(7)}
    or str(col).upper() in {f"PAYE_{i}" for i in range(7)}
]

if len(pay_cols) != 7:
    raise ValueError(f"Expected PAY_0 to PAY_6 or PAYE_0 to PAYE_6. Found: {pay_cols}")

# Arrange chronologically: PAY_6 is oldest, PAY_0 is most recent
pay_cols = sorted(pay_cols, key=lambda col: int(str(col).split("_")[-1]), reverse=True)

# Convert payment statuses into Markov states
# Negative values and zero represent no current delinquency
states = taiwan[pay_cols].copy().fillna(0).clip(lower=0)

# Build transition counts from consecutive payment-status observations
transition_counts = pd.DataFrame(0, index=range(0, 10), columns=range(0, 10))

for _, row in states.iterrows():
    for current_state, next_state in zip(row.iloc[:-1], row.iloc[1:]):
        current_state = min(int(current_state), 9)
        next_state = min(int(next_state), 9)
        transition_counts.loc[current_state, next_state] += 1

# Laplace smoothing and transition probabilities
transition_probabilities = (
    transition_counts + 1
).div((transition_counts + 1).sum(axis=1), axis=0)

# Default is defined as moving to a positive delinquency state
default_states = [state for state in transition_probabilities.columns if state > 0]

probability_of_default = transition_probabilities[default_states].sum(axis=1)

# Use the latest observed payment state to predict default probability
latest_state = states[pay_cols[-1]].astype(int).clip(upper=9)

taiwan["markov_default_probability"] = latest_state.map(probability_of_default)

# Predicted class using a 50% threshold
taiwan["markov_default_prediction"] = (
    taiwan["markov_default_probability"] >= 0.5
).astype(int)

print("Payment columns:", pay_cols)
print("\nTransition probabilities:")
display(transition_probabilities.round(3))

display(
    taiwan[
        pay_cols + [
            "markov_default_probability",
            "markov_default_prediction",
        ]
    ].head()
)

NameError: name 'taiwan' is not defined